In [0]:
employees_df = spark.read.csv(
    "/Volumes/workspace/default/my_volume/employees_noisy_5234.csv",
    header=True,
    inferSchema=True
)

In [0]:
from pyspark.sql.functions import(col, lower, trim, regexp_replace, when, to_date)

#Supprimer les doublons
employees_clean = employees_df.dropDuplicates()



In [0]:
from pyspark.sql.functions import lower, col, trim

columns_to_clean = [ "Department","Language", "Job_Title", "Seniority_level", "Education_Level", "Work_Location", "Newsletter_Subscription", "Decision_Maker_Flag","Preferred_Contact_Method", "Active_Flag", "Data_Source" ]

for c in columns_to_clean:
    employees_clean = employees_clean.withColumn(c, lower(trim(col(c))))

In [0]:
from pyspark.sql.functions import col, when, levenshtein, lit


# Colonne Language
employees_clean = employees_clean.withColumn(
    "Language",
    when(col("Language").isNull(), "Unknown")
    .when(levenshtein(col("Language"), lit("english")) <= 2, "English")
    .when(levenshtein(col("Language"), lit("turkish")) <= 2, "Turkish")
    .when(col("Language").contains("eng"), "English")
    .when(col("Language").contains("tur"), "Turkish")
    .otherwise("English")
)

In [0]:
from pyspark.sql.functions import col, lower, trim, levenshtein, lit, when, regexp_replace
from pyspark.sql import functions as F

# 1. Normalisation
employees_clean = employees_clean.withColumn(
    "Job_Title",
    lower(trim(col("Job_Title")))
)

# 2. Liste des valeurs propres (références)
reference_jobs = [
    "marketing specialist",
    "strategic buyer",
    "qa specialist",
    "digital marketing executive",
    "country manager",
    "mechanical engineer",
    "procurement manager",
    "quality assurance lead",
    "process engineer",
    "qa technician",
    "software developer",
    "electrical engineer",
    "brand manager",
    "account manager",
    "key account executive",
    "sales manager",
    "ops coordinator",
    "network engineer",
    "systems engineer",
    "procurement analyst",
    "operations director",
    "plant manager",
    "production engineer",
    "plant supervisor",
    "procurement specialist",
    "operations analyst",
    "logistics coordinator",
    "operations manager",
    "territory sales rep",
    "data centre operator",
    "production planner",
    "sales executive",
    "quality-by-design engineer",
    "infrastructure engineer",
    "growth marketer"
]

# 3. Trouver automatiquement la meilleure correspondance avec Levenshtein
best_match = None

for job in reference_jobs:
    current = F.struct(
        levenshtein(col("Job_Title"), lit(job)).alias("distance"),
        lit(job).alias("job")
    )

    if best_match is None:
        best_match = current
    else:
        best_match = F.least(best_match, current)

# 4. Remplacement par le métier le plus proche
employees_clean = employees_clean.withColumn(
    "Job_Title",
    F.initcap(best_match["job"])
)

# 5. Correction des acronymes et formats spéciaux
employees_clean = employees_clean.withColumn(
    "Job_Title",
    regexp_replace(col("Job_Title"), "Qa", "QA")
)

employees_clean = employees_clean.withColumn(
    "Job_Title",
    regexp_replace(
        col("Job_Title"),
        "Quality-By-Design",
        "Quality-by-Design"
    )
)

employees_clean = employees_clean.withColumn(
    "Job_Title",
    regexp_replace(
        col("Job_Title"),
        "Quality-by-design Engineer",
        "Quality-by-Design Engineer"
    )
)

# 6. Correction finale spécifique pour Plant Manager
employees_clean = employees_clean.withColumn(
    "Job_Title",
    when(
        col("Job_Title").rlike(
            "(?i)plantmanager|plant manager|pant manager|pxant manager|plnt manager"
        ),
        "Plant Manager"
    ).otherwise(col("Job_Title"))
)

# 7. Vérification finale
employees_clean.select("Job_Title").distinct().orderBy("Job_Title").show(100, False)

+---------------------------+
|Job_Title                  |
+---------------------------+
|Account Manager            |
|Brand Manager              |
|Country Manager            |
|Data Centre Operator       |
|Digital Marketing Executive|
|Electrical Engineer        |
|Growth Marketer            |
|Infrastructure Engineer    |
|Key Account Executive      |
|Logistics Coordinator      |
|Marketing Specialist       |
|Mechanical Engineer        |
|Network Engineer           |
|Operations Analyst         |
|Operations Director        |
|Operations Manager         |
|Ops Coordinator            |
|Plant Manager              |
|Plant Supervisor           |
|Process Engineer           |
|Procurement Analyst        |
|Procurement Manager        |
|Procurement Specialist     |
|Production Engineer        |
|Production Planner         |
|QA Specialist              |
|QA Technician              |
|Quality Assurance Lead     |
|Quality-by-Design Engineer |
|Sales Executive            |
|Sales Man

In [0]:
from pyspark.sql.functions import regexp_replace, col

employees_clean = employees_clean.withColumn(
    "Job_Title",
    regexp_replace(col("Job_Title"), "Qa", "QA")
)

employees_clean = employees_clean.withColumn(
    "Job_Title",
    regexp_replace(col("Job_Title"), "Quality-by-design", "Quality-by-Design")
)

In [0]:
from pyspark.sql.functions import col, lower, trim, when, levenshtein, lit, initcap

# 2. Liste des départements propres
reference_departments = [
    "marketing",
    "procurement",
    "quality",
    "management",
    "engineering",
    "it",
    "sales",
    "operations",
    "production"
]

# 3. Correction avec Levenshtein
expr = None

for dept in reference_departments:
    condition = levenshtein(col("Department"), lit(dept)) <= 2
    
    if expr is None:
        expr = when(condition, dept)
    else:
        expr = expr.when(condition, dept)

employees_clean = employees_clean.withColumn(
    "Department",
    expr.otherwise(col("Department"))
)

# 4. Mise en forme (Majuscule propre)
employees_clean = employees_clean.withColumn(
    "Department",
    initcap(col("Department"))
)

# 5. Correction spéciale IT (important)
from pyspark.sql.functions import regexp_replace

employees_clean = employees_clean.withColumn(
    "Department",
    regexp_replace(col("Department"), "It", "IT")
)

# 6. Vérification
display(employees_clean)

Employee_ID,Name,Department,Company_ID,Job_Title,Seniority_level,Education_Level,Work_Location,Newsletter_Subscription,Campaign_Response_Rate (%),Tenure_Years,Event_Attendance,Influence_Score,Decision_Maker_Flag,Preferred_Contact_Method,Language,Last_Contact_Date,Next_Followup_Date,Owner_Rep,Active_Flag,Data_Source
E00001,Z**** G*******,Marketing,C0015,Marketing Specialist,mid,master,office,no,27,5,3,61,no,email,Turkish,2024-04-06,2024-04-11,E**** T*******,yes,crm
E00006,L**** *******,Engineering,C0584,Mechanical Engineer,junior,phd,field,no,2,2,1,45,no,phone,Unknown,2024-08-16,2024-09-07,G**** R*******,yes,linkedin
E00028,M**** O*******,IT,C0330,Network Engineer,mid,bachelor,office,yes,3,5,2,23,no,phone,English,2024-01-10,2024-02-07,C**** D*******,yes,fair
E00031,O**** P*******,Procurement,C0428,Procurement Manager,junior,bachelor,factory,no,7,0,2,17,no,email,English,2024-06-21,2024-07-18,H**** N*******,yes,crm
E00061,Z**** Y*******,Marketing,C0553,Brand Manager,mid,bachelor,office,no,25,4,2,71,no,email,Turkish,2024-06-22,2024-07-22,E**** T*******,yes,crm
E00083,O**** Y*******,Production,C0279,Production Planner,junior,bachelor,office,no,5,2,0,47,no,phone,English,2024-05-25,2024-06-22,E**** T*******,yes,webinar
E00092,B**** Y*******,Engineering,C0595,Electrical Engineer,mid,bachelor,office,no,10,6,1,53,no,phone,Turkish,2024-03-25,2024-04-04,H**** N*******,yes,crm
E00099,B**** D*******,Operations,C0107,Logistics Coordinator,mid,master,office,yes,2,6,0,38,no,linkedin,English,2024-02-02,2024-02-16,A**** Y*******,yes,email campaign
E00122,T**** Y*******,Marketing,C0681,Digital Marketing Executive,mid,phd,remote,yes,43,2,2,46,no,email,Turkish,2024-01-04,2024-01-28,D**** S*******,yes,crm
E00125,S**** A*******,Procurement,C0309,Strategic Buyer,mid,master,office,yes,6,2,0,61,no,email,Turkish,2024-09-23,2024-10-06,F**** O*******,yes,linkedin


In [0]:
# Afficher toutes les valeurs uniques de Job_Title
employees_clean.select("Department").distinct().orderBy("Department").show(truncate=False, n=1000)

+-----------+
|Department |
+-----------+
|Engineering|
|IT         |
|Management |
|Marketing  |
|Operations |
|Procurement|
|Production |
|Quality    |
|Sales      |
+-----------+



In [0]:
# Preferred Contact Method
from pyspark.sql.functions import col, lower, trim, when

# 2. Mapping vers 4 valeurs uniquement
employees_clean = employees_clean.withColumn(
    "Preferred_Contact_Method",

    # Email
    when(col("Preferred_Contact_Method").rlike("mail|em"), "Email")

    # Phone
    .when(col("Preferred_Contact_Method").rlike("ph|fone|hone"), "Phone")

    # LinkedIn
    .when(col("Preferred_Contact_Method").rlike("link"), "LinkedIn")

    # Events
    .when(col("Preferred_Contact_Method").rlike("event|vent"), "Events")

    # Null / autres
    .otherwise("Email")
)

# 3. Vérification
employees_clean.select("Preferred_Contact_Method").distinct().show()

+------------------------+
|Preferred_Contact_Method|
+------------------------+
|                   Email|
|                   Phone|
|                LinkedIn|
|                  Events|
+------------------------+



In [0]:
from pyspark.sql.functions import col, lower, trim, when


# 2. Mapping strict
employees_clean = employees_clean.withColumn(
    "Data_Source",

    # CRM
    when(col("Data_Source").rlike("crm|^c.*m$|rm"), "CRM")

    # LinkedIn
    .when(col("Data_Source").rlike("link"), "LinkedIn")

    # Email Campaign
    .when(col("Data_Source").rlike("mail.*camp|email"), "Email Campaign")

    # Webinar
    .when(col("Data_Source").rlike("web|inar"), "Webinar")

    # Fair
    .when(col("Data_Source").rlike("fair|fai"), "Fair")

    # fallback (aucun unknown autorisé)
    .otherwise("Unknown")
)

# 3. Vérification
employees_clean.select("Data_Source").distinct().show()

+--------------+
|   Data_Source|
+--------------+
|           CRM|
|      LinkedIn|
|          Fair|
|       Webinar|
|Email Campaign|
|       Unknown|
+--------------+



In [0]:
from pyspark.sql.functions import col, initcap

# Liste des colonnes où tu veux appliquer initcap
columns_to_capitalize = [
    "Work_Location",
    "Education_Level",
    "Newsletter_Subscription",
    "Decision_Maker_Flag",
    "Active_Flag",
    "Seniority_level"
]

# Appliquer initcap seulement sur ces colonnes
for column_name in columns_to_capitalize:
    employees_clean = employees_clean.withColumn(
        column_name,
        initcap(col(column_name))
    )

# Vérification
display(employees_clean)

Employee_ID,Name,Department,Company_ID,Job_Title,Seniority_level,Education_Level,Work_Location,Newsletter_Subscription,Campaign_Response_Rate (%),Tenure_Years,Event_Attendance,Influence_Score,Decision_Maker_Flag,Preferred_Contact_Method,Language,Last_Contact_Date,Next_Followup_Date,Owner_Rep,Active_Flag,Data_Source
E00001,Z**** G*******,Marketing,C0015,Marketing Specialist,Mid,Master,Office,No,27,5,3,61,No,Email,Turkish,2024-04-06,2024-04-11,E**** T*******,Yes,CRM
E00006,L**** *******,Engineering,C0584,Mechanical Engineer,Junior,Phd,Field,No,2,2,1,45,No,Phone,Unknown,2024-08-16,2024-09-07,G**** R*******,Yes,LinkedIn
E00028,M**** O*******,IT,C0330,Network Engineer,Mid,Bachelor,Office,Yes,3,5,2,23,No,Phone,English,2024-01-10,2024-02-07,C**** D*******,Yes,Fair
E00031,O**** P*******,Procurement,C0428,Procurement Manager,Junior,Bachelor,Factory,No,7,0,2,17,No,Email,English,2024-06-21,2024-07-18,H**** N*******,Yes,CRM
E00061,Z**** Y*******,Marketing,C0553,Brand Manager,Mid,Bachelor,Office,No,25,4,2,71,No,Email,Turkish,2024-06-22,2024-07-22,E**** T*******,Yes,CRM
E00083,O**** Y*******,Production,C0279,Production Planner,Junior,Bachelor,Office,No,5,2,0,47,No,Phone,English,2024-05-25,2024-06-22,E**** T*******,Yes,Webinar
E00092,B**** Y*******,Engineering,C0595,Electrical Engineer,Mid,Bachelor,Office,No,10,6,1,53,No,Phone,Turkish,2024-03-25,2024-04-04,H**** N*******,Yes,CRM
E00099,B**** D*******,Operations,C0107,Logistics Coordinator,Mid,Master,Office,Yes,2,6,0,38,No,LinkedIn,English,2024-02-02,2024-02-16,A**** Y*******,Yes,Email Campaign
E00122,T**** Y*******,Marketing,C0681,Digital Marketing Executive,Mid,Phd,Remote,Yes,43,2,2,46,No,Email,Turkish,2024-01-04,2024-01-28,D**** S*******,Yes,CRM
E00125,S**** A*******,Procurement,C0309,Strategic Buyer,Mid,Master,Office,Yes,6,2,0,61,No,Email,Turkish,2024-09-23,2024-10-06,F**** O*******,Yes,LinkedIn


In [0]:
from pyspark.sql.functions import col, when, regexp_replace
from pyspark.sql.types import IntegerType

employees_clean = employees_clean.withColumn(
    "Campaign_Response_Rate (%)",
    regexp_replace(col("Campaign_Response_Rate (%)"), "%", "")
)

employees_clean = employees_clean.withColumn(
    "Campaign_Response_Rate (%)",
    when(col("Campaign_Response_Rate (%)").isNull(), 0)
    .otherwise(col("Campaign_Response_Rate (%)"))
    .cast(IntegerType())
)


In [0]:
employees_clean = employees_clean.withColumnRenamed(
    "Seniority_level",
    "Seniority_Level"
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

# 1. Nettoyage (comme tu as fait ✔)
employees_clean = employees_clean.withColumn(
    "Tenure_Years",
    F.when(
        F.col("Tenure_Years").rlike("^[0-9]+$"),
        F.col("Tenure_Years").cast("double")
    ).otherwise(None)
)

# 2. Fenêtre par Seniority_Level
window = Window.partitionBy("Seniority_Level")

# 3. Calcul médiane par groupe + imputation
employees_clean = employees_clean.withColumn(
    "Tenure_Years",
    F.when(
        F.col("Tenure_Years").isNull(),
        F.percentile_approx("Tenure_Years", 0.5).over(window)
    ).otherwise(F.col("Tenure_Years"))
)

# 4. Cast final en int
employees_clean = employees_clean.withColumn(
    "Tenure_Years",
    F.col("Tenure_Years").cast(IntegerType())
)

# 5. Vérification
employees_clean.select("Seniority_Level", "Tenure_Years") \
    .groupBy("Seniority_Level", "Tenure_Years") \
    .count() \
    .orderBy("Seniority_Level", "Tenure_Years") \
    .show(50)

+---------------+------------+-----+
|Seniority_Level|Tenure_Years|count|
+---------------+------------+-----+
|        C-level|          10|    4|
|        C-level|          11|    3|
|        C-level|          12|    5|
|        C-level|          13|    5|
|        C-level|          14|    3|
|        C-level|          15|    1|
|        C-level|          16|    4|
|        C-level|          17|    2|
|        C-level|          18|    2|
|        C-level|          19|    4|
|        C-level|          20|    7|
|        C-level|          21|    2|
|        C-level|          22|    4|
|        C-level|          23|    1|
|        C-level|          24|    3|
|       Director|           7|   35|
|       Director|           8|   37|
|       Director|           9|   39|
|       Director|          10|   54|
|       Director|          11|   36|
|       Director|          12|   52|
|       Director|          13|   34|
|       Director|          14|   32|
|       Director|          15|   28|
|

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

# 1. Remplacer 'N/A' et autres valeurs non-numériques par NULL
employees_clean = employees_clean.withColumn(
    "Event_Attendance",
    F.when(
        F.col("Event_Attendance").rlike("^[0-9]+$"),  # Seulement chiffres
        F.col("Event_Attendance").cast("double")
    ).otherwise(None)  # N/A, NULL, etc. → NULL
)

# 2. Fenêtre par Seniority_Level
window = Window.partitionBy("Seniority_Level")

# 3. Calculer médiane par groupe et imputer
employees_clean = employees_clean.withColumn(
    "Event_Attendance",
    F.when(
        F.col("Event_Attendance").isNull(),
        F.round(F.percentile_approx("Event_Attendance", 0.5).over(window))
    ).otherwise(F.col("Event_Attendance"))
)

# 4. Cast final en int
employees_clean = employees_clean.withColumn(
    "Event_Attendance",
    F.col("Event_Attendance").cast(IntegerType())
)

# 5. Vérification
employees_clean.select("Seniority_Level", "Event_Attendance") \
    .groupBy("Seniority_Level", "Event_Attendance") \
    .count() \
    .orderBy("Seniority_Level", "Event_Attendance") \
    .show(50)

+---------------+----------------+-----+
|Seniority_Level|Event_Attendance|count|
+---------------+----------------+-----+
|        C-level|               0|   12|
|        C-level|               1|   11|
|        C-level|               2|   18|
|        C-level|               3|    5|
|        C-level|               4|    2|
|        C-level|               5|    2|
|       Director|               0|  103|
|       Director|               1|  137|
|       Director|               2|  118|
|       Director|               3|    9|
|       Director|               4|   22|
|       Director|               5|   25|
|         Junior|               0|  366|
|         Junior|               1|  444|
|         Junior|               2|  431|
|         Junior|               3|   81|
|         Junior|               4|   82|
|         Junior|               5|   91|
|            Mid|               0|  444|
|            Mid|               1|  625|
|            Mid|               2|  606|
|            Mid

In [0]:
employees_clean.select("Event_Attendance").distinct().show()

+----------------+
|Event_Attendance|
+----------------+
|               0|
|               1|
|               2|
|               3|
|               4|
|               5|
+----------------+



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

# 1. Remplacer 'N/A' et autres valeurs non-numériques par NULL
employees_clean = employees_clean.withColumn(
    "Influence_Score",
    F.when(
        F.col("Influence_Score").rlike("^[0-9]+$"),  # Seulement chiffres
        F.col("Influence_Score").cast("double")
    ).otherwise(None)  # N/A, NULL, etc. → NULL
)

# 2. Fenêtre par Seniority_Level
window = Window.partitionBy("Seniority_Level")

# 3. Calculer médiane par groupe et imputer
employees_clean = employees_clean.withColumn(
    "Influence_Score",
    F.when(
        F.col("Influence_Score").isNull(),
        F.round(F.percentile_approx("Influence_Score", 0.5).over(window))
    ).otherwise(F.col("Influence_Score"))
)

# 4. Cast final en int
employees_clean = employees_clean.withColumn(
    "Influence_Score",
    F.col("Influence_Score").cast(IntegerType())
)

# 5. Vérification
employees_clean.select("Seniority_Level", "Influence_Score") \
    .groupBy("Seniority_Level") \
    .agg(
        F.min("Influence_Score").alias("min"),
        F.round(F.avg("Influence_Score"), 1).alias("moyenne"),
        F.max("Influence_Score").alias("max")
    ) \
    .orderBy("Seniority_Level") \
    .show()

+---------------+---+-------+---+
|Seniority_Level|min|moyenne|max|
+---------------+---+-------+---+
|        C-level| 33|   68.8| 98|
|       Director| 25|   68.3|100|
|         Junior|  9|   55.5|100|
|            Mid|  7|   55.3|100|
|         Senior| 19|   68.0|100|
+---------------+---+-------+---+



In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Étape 1 : normaliser la casse
employees_clean = employees_clean.withColumn(
    "Newsletter_Subscription",
    F.initcap(F.trim(F.lower(F.col("Newsletter_Subscription"))))
)

# Étape 2 : compter Yes et No par groupe
count_window = Window.partitionBy(
    "Seniority_Level", "Department", "Newsletter_Subscription"
)

employees_clean = employees_clean.withColumn(
    "val_count",
    F.count("Newsletter_Subscription").over(count_window)
)

# Étape 3 : trier par fréquence et prendre la valeur dominante
group_window = Window.partitionBy("Seniority_Level", "Department") \
                     .orderBy(F.desc("val_count"))

employees_clean = employees_clean.withColumn(
    "mode_val",
    F.first("Newsletter_Subscription", ignorenulls=True).over(group_window)
)

# Étape 4 : imputer les nulls
employees_clean = employees_clean.withColumn(
    "Newsletter_Subscription",
    F.when(
        F.col("Newsletter_Subscription").isNull(),
        F.col("mode_val")
    ).otherwise(F.col("Newsletter_Subscription"))
)

# Étape 5 : fallback mode global si groupe entier null
employees_clean = employees_clean.withColumn(
    "Newsletter_Subscription",
    F.when(
        F.col("Newsletter_Subscription").isNull(), "Yes"
    ).otherwise(F.col("Newsletter_Subscription"))
)

# Étape 6 : supprimer les colonnes temporaires
employees_clean = employees_clean.drop("val_count", "mode_val")

# Validation
employees_clean.groupBy("Newsletter_Subscription").count().show()

+-----------------------+-----+
|Newsletter_Subscription|count|
+-----------------------+-----+
|                     No| 2346|
|                    Yes| 2888|
+-----------------------+-----+



In [0]:
# Étape 1 : normaliser la casse
employees_clean = employees_clean.withColumn(
    "Owner_Rep",
    F.initcap(F.trim(F.lower(F.col("Owner_Rep"))))
)

# Étape 2 : calculer le mode par groupe directement
mode_by_group = (
    employees_clean
    .filter(F.col("Owner_Rep").isNotNull())
    .groupBy("Seniority_Level", "Department", "Owner_Rep")
    .count()
    .orderBy(F.desc("count"))
    .groupBy("Seniority_Level", "Department")
    .agg(F.first("Owner_Rep").alias("mode_val"))
)

# Étape 3 : joindre avec le dataset principal
employees_clean = employees_clean.join(
    mode_by_group,
    on=["Seniority_Level", "Department"],
    how="left"
)

# Étape 4 : imputer
employees_clean = employees_clean.withColumn(
    "Owner_Rep",
    F.when(
        F.col("Owner_Rep").isNull(),
        F.col("mode_val")
    ).otherwise(F.col("Owner_Rep"))
)

# Étape 5 : nettoyage
employees_clean = employees_clean.drop("mode_val")

# Validation
employees_clean.select("Owner_Rep").distinct().orderBy("Owner_Rep").show()

+--------------+
|     Owner_Rep|
+--------------+
|A**** Y*******|
|B**** K*******|
|C**** D*******|
|D**** S*******|
|E**** T*******|
|F**** O*******|
|G**** R*******|
|H**** N*******|
+--------------+



In [0]:
from pyspark.sql.functions import datediff, date_add, lit, when, col, to_date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Référence : première date du dataset
date_ref = lit("2024-01-01").cast("date")

# Étape 1 : remplacer 'unknown' et dates hors plage par null
employees_clean = employees_clean.withColumn(
    "Last_Contact_Date",
    when(
        F.lower(F.trim(col("Last_Contact_Date"))) == "unknown", None
    ).otherwise(col("Last_Contact_Date"))
)

# Étape 2 : parser en DateType
employees_clean = employees_clean.withColumn(
    "Last_Contact_Date",
    to_date(col("Last_Contact_Date"), "yyyy-MM-dd")
)

# Étape 3 : remplacer les dates hors plage par null
employees_clean = employees_clean.withColumn(
    "Last_Contact_Date",
    when(
        col("Last_Contact_Date") > lit("2024-09-29").cast("date"), None
    ).otherwise(col("Last_Contact_Date"))
)

# Étape 4 : convertir les dates en nombre de jours
employees_clean = employees_clean.withColumn(
    "days_since_ref",
    datediff(col("Last_Contact_Date"), date_ref)
)

# Étape 5 : calculer la médiane par groupe
window = Window.partitionBy("Seniority_Level", "Department")

employees_clean = employees_clean.withColumn(
    "median_days",
    F.percentile_approx("days_since_ref", 0.5).over(window)
)

# Étape 6 : imputer les nulls
employees_clean = employees_clean.withColumn(
    "Last_Contact_Date",
    when(
        col("Last_Contact_Date").isNull(),
        date_add(date_ref, col("median_days").cast("int"))
    ).otherwise(col("Last_Contact_Date"))
)

# Étape 7 : nettoyage colonnes temporaires
employees_clean = employees_clean.drop("days_since_ref", "median_days")

# Validation
print("=== Nulls résiduels (doit être 0) ===")
employees_clean.filter(col("Last_Contact_Date").isNull()).count()

print("\n=== Plage de dates ===")
employees_clean.select(
    F.min("Last_Contact_Date").alias("date_min"),
    F.max("Last_Contact_Date").alias("date_max")
).show()

=== Nulls résiduels (doit être 0) ===

=== Plage de dates ===
+----------+----------+
|  date_min|  date_max|
+----------+----------+
|2024-01-01|2024-09-29|
+----------+----------+



In [0]:
# Étape 1 : remplacer 'unknown' par null
employees_clean = employees_clean.withColumn(
    "Next_Followup_Date",
    F.when(
        F.lower(F.trim(F.col("Next_Followup_Date"))) == "unknown", None
    ).otherwise(F.col("Next_Followup_Date"))
)

# Étape 2 : parser en DateType
employees_clean = employees_clean.withColumn(
    "Next_Followup_Date",
    F.to_date(F.col("Next_Followup_Date"), "yyyy-MM-dd")
)

# Étape 3 : convertir en nombre de jours
date_ref = F.lit("2024-01-01").cast("date")

employees_clean = employees_clean.withColumn(
    "days_since_ref",
    F.datediff(F.col("Next_Followup_Date"), date_ref)
)

# Étape 4 : médiane par groupe
window = Window.partitionBy("Seniority_Level", "Department")

employees_clean = employees_clean.withColumn(
    "median_days",
    F.percentile_approx("days_since_ref", 0.5).over(window)
)

# Étape 5 : imputer les nulls
employees_clean = employees_clean.withColumn(
    "Next_Followup_Date",
    F.when(
        F.col("Next_Followup_Date").isNull(),
        F.date_add(date_ref, F.col("median_days").cast("int"))
    ).otherwise(F.col("Next_Followup_Date"))
)

# Étape 6 : nettoyage colonnes temporaires
employees_clean = employees_clean.drop("days_since_ref", "median_days")

# Validation
nulls = employees_clean.filter(F.col("Next_Followup_Date").isNull()).count()
print(f"Nulls résiduels : {nulls}")

employees_clean.select(
    F.min("Next_Followup_Date").alias("date_min"),
    F.max("Next_Followup_Date").alias("date_max")
).show()

Nulls résiduels : 0
+----------+----------+
|  date_min|  date_max|
+----------+----------+
|2024-01-03|2024-10-29|
+----------+----------+



In [0]:
# Étape 1 : normaliser la casse
employees_clean = employees_clean.withColumn(
    "Education_Level",
    F.initcap(F.trim(F.lower(F.col("Education_Level"))))
)

# Étape 2 : corriger PhD (initcap donne 'Phd' au lieu de 'PhD')
employees_clean = employees_clean.withColumn(
    "Education_Level",
    F.when(F.col("Education_Level") == "Phd", "PhD")
     .otherwise(F.col("Education_Level"))
)

# Étape 3 : compter les occurrences par groupe
count_window = Window.partitionBy(
    "Seniority_Level", "Department", "Education_Level"
)

employees_clean = employees_clean.withColumn(
    "val_count",
    F.count("Education_Level").over(count_window)
)

# Étape 4 : mode par groupe
group_window = Window.partitionBy("Seniority_Level", "Department") \
                     .orderBy(F.desc("val_count"))

employees_clean = employees_clean.withColumn(
    "mode_val",
    F.first("Education_Level", ignorenulls=True).over(group_window)
)

# Étape 5 : imputer les nulls
employees_clean = employees_clean.withColumn(
    "Education_Level",
    F.when(
        F.col("Education_Level").isNull(),
        F.col("mode_val")
    ).otherwise(F.col("Education_Level"))
)

# Étape 6 : fallback mode global
employees_clean = employees_clean.withColumn(
    "Education_Level",
    F.when(
        F.col("Education_Level").isNull(), "Bachelor"
    ).otherwise(F.col("Education_Level"))
)

# Étape 7 : nettoyage
employees_clean = employees_clean.drop("val_count", "mode_val")

# Validation
employees_clean.groupBy("Education_Level").count().orderBy(F.desc("count")).show()

+---------------+-----+
|Education_Level|count|
+---------------+-----+
|       Bachelor| 2870|
|         Master| 1560|
|    High School|  531|
|            PhD|  273|
+---------------+-----+



In [0]:
# Vérifier les nulls sur toutes les colonnes en une seule fois
from pyspark.sql.functions import col, count, when

null_counts = employees_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in employees_clean.columns
])

null_counts.show(vertical=True)

-RECORD 0-------------------------
 Seniority_Level            | 0   
 Department                 | 0   
 Employee_ID                | 0   
 Name                       | 0   
 Company_ID                 | 0   
 Job_Title                  | 0   
 Education_Level            | 0   
 Work_Location              | 0   
 Newsletter_Subscription    | 0   
 Campaign_Response_Rate (%) | 0   
 Tenure_Years               | 0   
 Event_Attendance           | 0   
 Influence_Score            | 0   
 Decision_Maker_Flag        | 0   
 Preferred_Contact_Method   | 0   
 Language                   | 0   
 Last_Contact_Date          | 0   
 Next_Followup_Date         | 0   
 Owner_Rep                  | 0   
 Active_Flag                | 0   
 Data_Source                | 0   



In [0]:
# 1. Vérifier les types de toutes les colonnes
employees_clean.printSchema()

root
 |-- Seniority_Level: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Employee_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Company_ID: string (nullable = true)
 |-- Job_Title: string (nullable = false)
 |-- Education_Level: string (nullable = true)
 |-- Work_Location: string (nullable = true)
 |-- Newsletter_Subscription: string (nullable = true)
 |-- Campaign_Response_Rate (%): integer (nullable = true)
 |-- Tenure_Years: integer (nullable = true)
 |-- Event_Attendance: integer (nullable = true)
 |-- Influence_Score: integer (nullable = true)
 |-- Decision_Maker_Flag: string (nullable = true)
 |-- Preferred_Contact_Method: string (nullable = false)
 |-- Language: string (nullable = false)
 |-- Last_Contact_Date: date (nullable = true)
 |-- Next_Followup_Date: date (nullable = true)
 |-- Owner_Rep: string (nullable = true)
 |-- Active_Flag: string (nullable = true)
 |-- Data_Source: string (nullable = false)



In [0]:
# 2. Vérifier les valeurs uniques des colonnes catégorielles
for col in ["Seniority_Level", "Work_Location", "Active_Flag",
            "Decision_Maker_Flag", "Newsletter_Subscription",
            "Education_Level", "Department"]:
    print(f"\n=== {col} ===")
    employees_clean.groupBy(col).count().orderBy(F.desc("count")).show()


=== Seniority_Level ===
+---------------+-----+
|Seniority_Level|count|
+---------------+-----+
|            Mid| 1980|
|         Junior| 1495|
|         Senior| 1295|
|       Director|  414|
|        C-level|   50|
+---------------+-----+


=== Work_Location ===
+-------------+-----+
|Work_Location|count|
+-------------+-----+
|       Office| 3401|
|      Factory|  804|
|       Remote|  525|
|        Field|  504|
+-------------+-----+


=== Active_Flag ===
+-----------+-----+
|Active_Flag|count|
+-----------+-----+
|        Yes| 4833|
|         No|  401|
+-----------+-----+


=== Decision_Maker_Flag ===
+-------------------+-----+
|Decision_Maker_Flag|count|
+-------------------+-----+
|                 No| 3783|
|                Yes| 1451|
+-------------------+-----+


=== Newsletter_Subscription ===
+-----------------------+-----+
|Newsletter_Subscription|count|
+-----------------------+-----+
|                    Yes| 2888|
|                     No| 2346|
+-----------------------+

In [0]:
employees_clean.select(
    "Tenure_Years",
    "Event_Attendance",
    "Influence_Score",
    "Campaign_Response_Rate (%)"
).describe().show()

+-------+------------------+------------------+------------------+--------------------------+
|summary|      Tenure_Years|  Event_Attendance|   Influence_Score|Campaign_Response_Rate (%)|
+-------+------------------+------------------+------------------+--------------------------+
|  count|              5234|              5234|              5234|                      5234|
|   mean| 4.894917844860528|  1.53821169277799| 59.65322888803974|        7.2244936950706915|
| stddev|3.8399621792985412|1.3325983622525714|17.404450243887453|         8.241658916656775|
|    min|                 0|                 0|                 7|                         0|
|    max|                24|                 5|               100|                        47|
+-------+------------------+------------------+------------------+--------------------------+



In [0]:
import os

# Étape 1 : sauvegarder
employees_clean.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/my_volume/employees_clean_temp")

# Étape 2 : trouver le fichier part-00000
files = dbutils.fs.ls("/Volumes/workspace/default/my_volume/employees_clean_temp")
part_file = [f.path for f in files if f.name.startswith("part-")][0]

# Étape 3 : renommer vers le nom souhaité
dbutils.fs.cp(
    part_file,
    "/Volumes/workspace/default/my_volume/employees_clean_final.csv"
)

# Étape 4 : supprimer le dossier temporaire
dbutils.fs.rm("/Volumes/workspace/default/my_volume/employees_clean_temp", recurse=True)

# Validation
display(dbutils.fs.ls("/Volumes/workspace/default/my_volume/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/my_volume/companies_clean_final.csv,companies_clean_final.csv,121559,1778112240000
dbfs:/Volumes/workspace/default/my_volume/companies_noisy_734.csv,companies_noisy_734.csv,122165,1776509821000
dbfs:/Volumes/workspace/default/my_volume/employees_clean_final/,employees_clean_final/,0,1778112511070
dbfs:/Volumes/workspace/default/my_volume/employees_clean_final.csv,employees_clean_final.csv,815775,1778112511000
dbfs:/Volumes/workspace/default/my_volume/employees_noisy_5234.csv,employees_noisy_5234.csv,818844,1776509821000
